In [62]:
%load_ext autoreload
%autoreload 2 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [63]:
import pandas as pd
import math

import plotly.express as px
import plotly.io as pio   
pio.kaleido.scope.mathjax = None

In [64]:
# Reduced = Only models in Table 4
detection_df = pd.read_csv(
    "/home/gquetel/experiences-results/2025-12-05-all-results/results-merged.csv"
)
inference_df = pd.read_csv(
    "/home/gquetel/experiences-results/2025-12-05-all-results/inference-merged.csv"
)
n_samples = 3353671

In [65]:
def rename_models(df) -> pd.DataFrame:
    df["display name"] = df["model"]
    # manual_positions = {
    #     "GAUR and OCSVM-scaler-claude": "OCSVM (Claude)",
    #     "GAUR and AE-scaler-claude": "AE (Claude)",
    #     "GAUR and OCSVM-scaler-ruleid": "OCSVM (ruleid)",
    #     "GAUR and AE-scaler-ruleid": "AE (ruleid)",
    #     "GAUR and OCSVM-scaler-chatgpt": "OCSVM (ChatGPT)",
    #     "GAUR and AE-scaler-chatgpt": "AE (ChatGPT)",
    #     "GAUR and OCSVM-scaler-gpt-oss": "OCSVM (GPT-OSS)",
    #     "GAUR and AE-scaler-gpt-oss": "AE (GPT-OSS)",
    #     "Li and AE-scaler": "AE (Li)",
    #     "Li and OCSVM-scaler": "OCSVM (Li)",
    #     "GAUR and OCSVM-scaler-expert": "OCSVM (Expert)",
    #     "GAUR and AE-scaler-expert": "AE (Expert)",
    #     "SecureBERT and AE": "AE (SecureBERT)",
    #     "SecureBERT and OCSVM": "OCSVM (SecureBERT)",
    #     "GAUR and OCSVM-scaler-llama": "OCSVM (Llama)",
    #     "GAUR and AE-scaler-llama": "AE (Llama)",
    #     "GAUR and OCSVM-scaler-mistral": "OCSVM (Mistral)",
    #     "GAUR and AE-scaler-mistral": "AE (Mistral)",
    # }


    manual_positions = {
        "GAUR and OCSVM-scaler-claude": "OCSVM ",
        "GAUR and AE-scaler-claude": "AE",
        "GAUR and OCSVM-scaler-ruleid": "OCSVM",
        "GAUR and AE-scaler-ruleid": "AE",
        "GAUR and OCSVM-scaler-chatgpt": "OCSVM",
        "GAUR and AE-scaler-chatgpt": "AE",
        "GAUR and OCSVM-scaler-gpt-oss": "OCSVM",
        "GAUR and AE-scaler-gpt-oss": "AE",
        "Li and AE-scaler": "AE",
        "Li and OCSVM-scaler": "OCSVM",
        "GAUR and OCSVM-scaler-expert": "OCSVM",
        "GAUR and AE-scaler-expert": "AE",
        "SecureBERT and AE": "AE",
        "SecureBERT and OCSVM": "OCSVM",
        "GAUR and OCSVM-scaler-llama": "OCSVM",
        "GAUR and AE-scaler-llama": "AE",
        "GAUR and OCSVM-scaler-mistral": "OCSVM",
        "GAUR and AE-scaler-mistral": "AE",
    }

    df["display name"] = df["display name"].replace(manual_positions)
    return df

In [66]:
detection_df["rocauc"] = detection_df["rocauc"].astype(float)

def parse_timedelta(td_str):
    # td_str is like '0 days 00:01:58.972381'
    h, m, s = td_str.split()[2].split(":")
    d = 86400 * int(td_str.split()[0])
    seconds = d + int(h) * 3600 + int(m) * 60 + float(s)
    return seconds

# We divide by number of sample to get avg time to process a query
inference_df["inference_seconds"] = (
    inference_df["inference_time"].apply(parse_timedelta) / n_samples
)
merged_df = pd.merge(detection_df, inference_df, on="model")

# Define Model type with the GAUR skind rule
def assign_model_type(row):
    # Should ehtier return: expert, Claude, ChatGPT, Llama, Mistral OSS-GPT, rule-id, Li et al. or SecureBERT
    name = row["model"]

    if name.startswith("SecureBERT") or name.startswith("Secure-BERT"):
        return "SecureBERT (with GPU)"

    if name.startswith("Li "):
        return "Li et al."

    if "scaler-" in name:
        trace_type = name.split("scaler-")[1]
        match trace_type:
            case "claude":
                return "Claude"
            case "chatgpt":
                return "ChatGPT"
            case "llama":
                return "Llama"
            case "mistral":
                return "Mistral"
            case "gpt-oss":
                return "OSS-GPT"
            case "ruleid":
                return "rule-id"
            case "expert":
                return "expert"
            case _:
                raise ValueError("Unknown trace: ", trace_type)
    raise ValueError("Unknown model type:", name)


merged_df["Model type"] = merged_df.apply(assign_model_type, axis=1)
merged_df = rename_models(merged_df)
# display(merged_df)

In [ ]:
manual_position_offsets = {
    "GAUR and OCSVM-scaler-ruleid": (20, -20),
    "GAUR and AE-scaler-ruleid": (20, -20),
    "SecureBERT and AE": (20, -20),
    "SecureBERT and OCSVM": (20, -20),
    "Li and OCSVM-scaler": (20, -20),
    "Li and AE-scaler": (-20, 20),
}

merged_df["position_offset"] = merged_df["model"].map(manual_position_offsets)

fig = px.scatter(
    merged_df,
    x="rocauc",
    y="inference_seconds",
    color="Model type",
    log_y=True,
    color_discrete_sequence=[
        "#DCE775",
        "#E57373",
        "#4FC3F7",
        "#8b7e74",
        "#9fa8da",
        "#66BB6A",
        "#0D47A1",
        "#ffd54f",
        "#7E57C2",
    ],
    # symbol="marker_symbol", #  a value that separates the data categories from eachother
    # symbol_sequence = ['circle', 'square', 'x', 'cross'],
    # title="AUROC vs Inference Time",
    labels={"rocauc": "AUROC", "inference_seconds": "Inference time per query (s)"},
    height=700,
    width=900,
    template="plotly_white",
    category_orders={
        "Model type": [
            "expert",
            "Claude",
            "ChatGPT",
            "Llama",
            "Mistral",
            "OSS-GPT",
            "rule-id",
            "Li et al.",
            "SecureBERT (with GPU)",
        ]
    },
)
#


fig.update_traces(
    marker=dict(
        size=20,
        symbol="square",
        line=dict(
            width=1,
            color="#3b3b3b",
        ),
    ),
    selector=dict(mode="markers"),
)

fig.update_layout(
    legend=dict(
        orientation="v",
        y=0.015,
        x=0.015,
        title=dict(text="\tFeature extractor"),
        bgcolor="rgba(255, 255, 255, 0.5)",
        bordercolor="#3b3b3b",
        borderwidth=1,
        font=dict(size=18, family="Arial", color="#3b3b3b"),
    ),
    xaxis=dict(
        title_font=dict(size=24, family="Arial"),  # x-axis label
        tickfont=dict(size=18),  # x-axis tick labels
        ticksuffix="",
    ),
    yaxis=dict(
        title_font=dict(size=24, family="Arial"),  # y-axis label
        tickfont=dict(size=15),  # y-axis tick labels
    ),
)

# Annotation for group of models:
fig.update_layout(
    shapes=[
        # OCSVM
        dict(
            # Placed Relative to the Axis Position and Length
            type="circle",
            xref="x domain",
            yref="y domain",
            x0=0.72,
            x1=0.83,
            y0=0.42,
            y1=0.52,
        ),
        # AE
        dict(
            # Placed Relative to the Axis Position and Length
            type="circle",
            xref="x domain",
            yref="y domain",
            x0=0.82,
            x1=0.94,
            y0=0.24,
            y1=0.11,
        ),
    ]
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=(0.72 + 0.83) / 2,
    y=0.52,
    text="OCSVM",
    ax=20,  # Offset en pixel
    ay=-20,
    showarrow=True,  # shows the connecting line
    arrowhead=0,
    font=dict(size=18, color="#3b3b3b"),
    arrowsize=1,
    arrowwidth=2,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=(0.82 + 0.94) / 2,
    y=0.24,
    text="AE",
    ax=20,  # Offset en pixel
    ay=-20,
    showarrow=True,  # shows the connecting line
    arrowhead=0,
    font=dict(size=18, color="#3b3b3b"),
    arrowsize=1,
    arrowwidth=2,
)


# Add annotations with connecting lines
for i, row in merged_df.iterrows():
    pos = row["position_offset"]
    if isinstance(pos, float):
        continue  # is NAN
    ax, ay = row["position_offset"]
    fig.add_annotation(
        x=row["rocauc"],
        y=math.log10(row["inference_seconds"]),
        text=row["display name"],
        showarrow=True,  # shows the connecting line
        arrowhead=0,
        ax=ax,  # Offset en pixel
        ay=ay,
        font=dict(size=18),
        arrowcolor="#3b3b3b",
        arrowsize=1,
        arrowwidth=0.1,
    )
fig.write_image("plot-auroc.pdf")
fig.show()

In [68]:
#  "fone" is of the form "XX.XX%", we need to transform this into float to have a correct figure.
def remove_model(df, names):
    if isinstance(names, str):
        names = [names]
    df = df[~df["model"].isin(names)].copy()
    return df


merged_df["fone"] = merged_df["fone"].str.rstrip("%").astype(float)
manual_position_offsets = {
    "GAUR and OCSVM-scaler-ruleid": (20, -20),
    "GAUR and AE-scaler-ruleid": (-20, 20),
    "SecureBERT and AE": (-20, -20),
    "Li and OCSVM-scaler": (20, -20),
    "Li and AE-scaler": (20, -20),
}

merged_df["position_offset"] = merged_df["model"].map(manual_position_offsets)
merged_df_1 = remove_model(merged_df, ["SecureBERT and OCSVM"])


fig = px.scatter(
    merged_df_1,
    x="fone",
    y="inference_seconds",
    color="Model type",
    log_y=True,
    color_discrete_sequence=[
        "#E57373",
        "#6490f6",
        "#4FC3F7",
        "#66BB6A",
        "#ffd54f",
        "#DCE775",
        "#7E57C2",
        "#8b7e74",
        "#9fa8da",
    ],
    labels={"fone": "F1 Score", "inference_seconds": "Inference time per query (s)"},
    height=600,
    width=1200,
    template="plotly_white",
)

fig.update_traces(
    marker=dict(
        size=20,
        symbol="square",
        line=dict(
            width=1,
            color="#3b3b3b",
        ),
    ),
    selector=dict(mode="markers"),
)

fig.update_layout(
    legend=dict(
        orientation="v",
        # y=0,
        # x=0,
        title=dict(text="Feature extractor"),
        bgcolor="rgba(255, 255, 255, 0.5)",
        bordercolor="#3b3b3b",
        borderwidth=1,
        font=dict(size=18, family="Arial", color="#3b3b3b"),

    ),
    xaxis=dict(
        title_font=dict(size=20, family="Arial"),  # x-axis label
        tickfont=dict(size=18),  # x-axis tick labels
        ticksuffix="%",
    ),
    yaxis=dict(
        title_font=dict(size=20, family="Arial"),  # y-axis label
        tickfont=dict(size=15),  # y-axis tick labels
    ),
)
# Annotation for group of models:
fig.update_layout(
    shapes=[
        # OCSVM
        dict(
            # Placed Relative to the Axis Position and Length
            type="circle",
            xref="x domain",
            yref="y domain",
            x0=0.86,
            x1=0.96,
            y0=0.48,
            y1=0.58,
        ),

        # AE
        dict(
            # Placed Relative to the Axis Position and Length
            type="circle",
            xref="x domain",
            yref="y domain",
            x0=0.86,
            x1=0.97,
            y0=0.24,
            y1=0.13,
        )
    ]
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=(0.86 + 0.96) / 2,
    y=0.58,
    text="OCSVM",
    ax=20,  # Offset en pixel
    ay=-20,
    showarrow=True,  # shows the connecting line
    arrowhead=0,
    font=dict(size=18, color="#3b3b3b"),
    arrowsize=1,
    arrowwidth=2,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=(0.86 + 0.97) / 2,
    y=0.24,
    text="AE",
    ax=20,  # Offset en pixel
    ay=-20,
    showarrow=True,  # shows the connecting line
    arrowhead=0,
    font=dict(size=18, color="#3b3b3b"),
    arrowsize=1,
    arrowwidth=2,
)


# Annotations regarding model types.
for i, row in merged_df_1.iterrows():
    pos = row["position_offset"]
    if isinstance(pos, float):  # means NAN
        continue

    ax, ay = row["position_offset"]
    fig.add_annotation(
        x=row["fone"],
        y=math.log10(row["inference_seconds"]),
        text=row["display name"],
        showarrow=True,  # shows the connecting line
        arrowhead=0,
        ax=ax,  # Offset en pixel
        ay=ay,
        font=dict(size=18),
        arrowcolor="#3b3b3b",
        arrowsize=1,
        arrowwidth=0.1,
    )
fig.write_image("plot.pdf")
fig.show()

In [69]:
# On veut garder: SecureBERT AE, Li *2, ChatGPT * 2, Claude AE, ruleid * 2
# On veut dégager:
dropped_models = [
    "SecureBERT and OCSVM",
    "GAUR and OCSVM-scaler-gpt-oss",
    "GAUR and AE-scaler-gpt-oss",
    "GAUR and OCSVM-scaler-expert",
    "GAUR and AE-scaler-expert",
    "GAUR and OCSVM-scaler-llama",
    "GAUR and AE-scaler-llama",
    "GAUR and OCSVM-scaler-mistral",
    "GAUR and AE-scaler-mistral",
    "GAUR and OCSVM-scaler-claude",
]

merged_df_2 = remove_model(merged_df, dropped_models)

manual_position_offsets = {
    "GAUR and OCSVM-scaler-ruleid": (20, -20),
    "GAUR and AE-scaler-ruleid": (20, -20),
    "SecureBERT and AE": (20, -20),
    "Li and OCSVM-scaler": (20, -20),
    "Li and AE-scaler": (20, -20),
    "GAUR and OCSVM-scaler-chatgpt": (20, -20),
    "GAUR and AE-scaler-chatgpt": (-20, -20),
    "GAUR and AE-scaler-claude": (20, -20),
}

merged_df_2["position_offset"] = merged_df_2["model"].map(manual_position_offsets)


fig = px.scatter(
    merged_df_2,
    x="fone",
    y="inference_seconds",
    color="Model type",
    log_y=True,
    color_discrete_sequence=["#E57373", "#6490f6", "#4FC3F7", "#ffd54f", "#7E57C2"],
    labels={"fone": "F1 Score", "inference_seconds": "Inference time per query (s)"},
    height=600,
    width=1200,
    template="plotly_white",
)
#

fig.update_traces(
    marker=dict(
        size=12,
        symbol="square",
        line=dict(
            width=1,
            color="#3b3b3b",
        ),
    ),
    selector=dict(mode="markers"),
)

fig.update_layout(
    legend=dict(
        font=dict(size=18, family="Arial", color="#3b3b3b"),
        title=dict(
            text="Feature extractor",
            font=dict(size=20),  # Separate size for legend title
        ),
        orientation="v",
        y=1,
        x=0.015,
        # Add opacity/transparency to legend background
        bgcolor="rgba(255, 255, 255, 0.5)",
        bordercolor="#3b3b3b",
        borderwidth=1,
    ),
    xaxis=dict(
        title_font=dict(size=20, family="Arial"),  # x-axis label
        tickfont=dict(size=18),  # x-axis tick labels
        ticksuffix="%",
    ),
    yaxis=dict(
        title_font=dict(size=20, family="Arial"),  # y-axis label
        tickfont=dict(size=15),  # y-axis tick labels
    ),
)

# Add annotations with connecting lines
for i, row in merged_df_2.iterrows():
    ax, ay = row["position_offset"]
    fig.add_annotation(
        x=row["fone"],
        y=math.log10(row["inference_seconds"]),
        text=row["display name"],
        showarrow=True,  # shows the connecting line
        arrowhead=0,
        ax=ax,  # Offset en pixel
        ay=ay,
        font=dict(size=18),
        arrowcolor="#3b3b3b",
        arrowsize=1,
        arrowwidth=2,
    )
fig.show()